# Hyperparameter Tuning in Machine Learning

Techniques covered:
1. **Grid Search** — exhaustively tries every combination
2. **Random Search** — randomly samples combinations

**Dataset:** Iris | **Model:** SVM | **Tuned params:** C, kernel, gamma

In [ ]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import loguniform

## Data

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Hyperparameter Spaces

In [ ]:
# Grid Search needs explicit lists
grid_param_space = {
    "C":      [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly"],
    "gamma":  ["scale", "auto"],
}

# Random Search can use lists OR statistical distributions
random_param_space = {
    "C":      loguniform(0.1, 100),
    "kernel": ["linear", "rbf", "poly"],
    "gamma":  ["scale", "auto"],
}

## 1. Grid Search
Tries **all** combinations: 4 × 3 × 2 = **24 fits** (× 5 CV folds = 120 total)

In [ ]:
grid_search = GridSearchCV(
    estimator=SVC(),
    param_grid=grid_param_space,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f"Best params : {grid_search.best_params_}")
print(f"Best CV acc : {grid_search.best_score_:.4f}")
print(f"Test acc    : {accuracy_score(y_test, grid_search.predict(X_test)):.4f}")

## 2. Random Search
Tries only **n_iter=20** randomly sampled combinations — much faster for large spaces

In [ ]:
random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=random_param_space,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
)
random_search.fit(X_train, y_train)

print(f"Best params : {random_search.best_params_}")
print(f"Best CV acc : {random_search.best_score_:.4f}")
print(f"Test acc    : {accuracy_score(y_test, random_search.predict(X_test)):.4f}")

## Comparison

In [ ]:
print(f"{'Technique':<15} {'Combinations tried':>20} {'Test Accuracy':>15}")
print("-" * 52)
print(f"{'Grid Search':<15} {len(grid_search.cv_results_['params']):>20} "
      f"{accuracy_score(y_test, grid_search.predict(X_test)):>15.4f}")
print(f"{'Random Search':<15} {len(random_search.cv_results_['params']):>20} "
      f"{accuracy_score(y_test, random_search.predict(X_test)):>15.4f}")